# 沪深 300：历年日 K 线

用 AkShare 获取沪深 300（`sh000300`）在数据源中可用的全部历史日线，绘制开、高、低、收蜡烛图。

- 数据接口：[`stock_zh_index_daily`](https://akshare.akfamily.xyz/data/index/index.html)，来源：新浪财经。
- 指数点位，不做个股前复权或后复权；每根蜡烛代表一个交易日。
- 红涨绿跌；下方显示成交量（保留接口原始口径）。
- 图表默认显示全历史，可用按钮、底部滑块或拖动缩放查看某段时间，不会转换成周 K/月 K。
- 历史起点和最新日期以接口实际返回为准；盘中当日数据可能尚未收盘。

先在项目虚拟环境中执行 `python -m pip install -e '.[notebook]'`，并选择该环境的 Python 内核，然后从上到下运行。此 Notebook 每次执行获取单元格都会重新联网。


In [1]:
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

import akshare as ak
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

# 支持从项目根目录或 notebooks/ 启动，不修改 sys.path。
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
OUTPUT_DIR = ROOT / "outputs"
RAW_DIR = ROOT / "data" / "raw"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)
print(f"AkShare {ak.__version__}")


AkShare 1.18.88


## 获取与检查数据

不预设历史起点；检查日期、重复记录、空值以及 OHLC 的基本关系，避免把异常行情画成正常 K 线。

In [2]:
SYMBOL = "sh000300"
raw = ak.stock_zh_index_daily(symbol=SYMBOL)
retrieved_at = datetime.now(ZoneInfo("Asia/Shanghai")).isoformat(timespec="seconds")
if raw.empty:
    raise ValueError("接口返回空数据，请检查数据源与网络。")

required = ["date", "open", "high", "low", "close", "volume"]
missing = set(required) - set(raw.columns)
if missing:
    raise ValueError(f"接口缺少字段：{missing}")

df = raw[required].copy()
df["date"] = pd.to_datetime(df["date"], errors="raise")
for column in required[1:]:
    df[column] = pd.to_numeric(df[column], errors="raise")
df = df.sort_values("date").reset_index(drop=True)
assert not df.isna().any().any(), "数据含缺失值"
assert not df["date"].duplicated().any(), "存在重复交易日"
assert (df[["open", "high", "low", "close"]] > 0).all().all(), "指数点位应大于零"
assert (df["high"] >= df[["open", "close", "low"]].max(axis=1)).all(), "最高价异常"
assert (df["low"] <= df[["open", "close", "high"]].min(axis=1)).all(), "最低价异常"
assert (df["volume"] >= 0).all(), "成交量异常"

raw.to_csv(RAW_DIR / "csi300_daily_sina.csv", index=False)
print(f"获取时间：{retrieved_at}")
print(f"数据范围：{df.date.min():%Y-%m-%d} 至 {df.date.max():%Y-%m-%d}，共 {len(df):,} 个交易日")
display(df.head(3), df.tail(3))


获取时间：2026-11-06T23:45:05+08:00
数据范围：2002-01-04 至 2026-11-06，共 6,025 个交易日


,date,open,high,low,close,volume
0,2002-01-04,1316.455,1316.455,1316.455,1316.455,0
1,2002-01-07,1302.084,1302.084,1302.084,1302.084,0
2,2002-01-08,1292.714,1292.714,1292.714,1292.714,0


,date,open,high,low,close,volume
6022,2026-11-04,4007.351,4105.858,3987.152,4101.335,24637197500
6023,2026-11-05,4065.724,4136.798,4065.724,4113.411,24197994600
6024,2026-11-06,4017.940,4069.072,4013.793,4054.277,19215201200


## 交互式日 K 线与成交量

全历史数据较密集，可选择 1 年或 3 年看清蜡烛；日历坐标会保留周末和节假日间隔。图表同时导出为可离线打开的 HTML。

In [3]:
fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    vertical_spacing=0.04, row_heights=[0.78, 0.22],
)
fig.add_trace(go.Candlestick(
    x=df["date"], open=df["open"], high=df["high"],
    low=df["low"], close=df["close"], name="沪深 300 日 K",
    increasing_line_color="#e5484d", decreasing_line_color="#16a085",
), row=1, col=1)
colors = ["#e5484d" if close >= opening else "#16a085"
          for opening, close in zip(df["open"], df["close"])]
fig.add_trace(go.Bar(
    x=df["date"], y=df["volume"], marker_color=colors, name="成交量（原始口径）",
), row=2, col=1)
fig.update_layout(
    title=f"沪深 300 · 历年日 K | {df.date.min():%Y-%m-%d} — {df.date.max():%Y-%m-%d}",
    template="plotly_white", height=780, showlegend=False,
    margin=dict(l=70, r=30, t=100, b=50),
    font=dict(family="Noto Sans CJK SC, Microsoft YaHei, sans-serif"),
)
fig.update_yaxes(title_text="指数点位", fixedrange=False, row=1, col=1)
fig.update_yaxes(title_text="成交量", rangemode="tozero", row=2, col=1)
fig.update_xaxes(rangeslider_visible=False, row=1, col=1)
fig.update_xaxes(
    rangeselector=dict(buttons=[
        dict(count=6, label="6个月", step="month", stepmode="backward"),
        dict(count=1, label="1年", step="year", stepmode="backward"),
        dict(count=3, label="3年", step="year", stepmode="backward"),
        dict(count=1, label="今年", step="year", stepmode="todate"),
        dict(label="全部", step="all"),
    ]), row=1, col=1,
)
fig.update_xaxes(rangeslider=dict(visible=True, thickness=0.10), row=2, col=1)
html_path = OUTPUT_DIR / "csi300_daily_kline.html"
fig.write_html(html_path, include_plotlyjs=True, full_html=True)
print(f"离线交互图：{html_path}")
fig.show(renderer="plotly_mimetype")


离线交互图：/home/yutian/文档/ChatGPT/TradingAgent/outputs/csi300_daily_kline.html
